In [14]:
long_document = """
University Academic Handbook

Attendance Policy:
Students must maintain at least 75% attendance in every subject to be eligible for semester examinations.
Students with attendance below 75% may not be allowed to write final exams unless special approval is granted.

Hostel Rules:
Hostel gates close at 9 PM on weekdays and 10 PM on weekends.
Students must carry their ID cards while entering the hostel.

Fee Refund Policy:
The refund deadline for semester fees is 15 days after the start of classes.
No refund will be issued after this deadline except under extraordinary circumstances.

Library Policy:
Students can borrow up to 4 books at a time.
Books must be returned within 14 days to avoid a fine.

Examination Rules:
Students must carry their hall ticket to the exam hall.
Electronic devices are not allowed during exams.
"""

In [29]:
evaluation_data = [
    {
        "query": "What attendance is required for semester exams?",
        "expected_keyword": "75% attendance"
    },
    {
        "query": "When do hostel gates close?",
        "expected_keyword": "9 PM on weekdays"
    },
    {
        "query": "What is the refund deadline for fees?",
        "expected_keyword": "15 days after the start of classes"
    },
    {
        "query": "How many books can students borrow?",
        "expected_keyword": "4 books"
    },
    {
        "query": "Are electronic devices allowed in exams?",
        "expected_keyword": "Electronic devices are not allowed"
    }
]

In [17]:
def section_aware_chunking(text):
    chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]
    return chunks

In [27]:
section_chunks = section_aware_chunking(long_document)

In [30]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load models
minilm_model = SentenceTransformer("all-MiniLM-L6-v2")
bge_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# FAISS builder
def build_faiss_index_from_embeddings(embeddings):
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

# Embedding helper
def get_embeddings(texts, model):
    embeddings = model.encode(texts)
    return np.array(embeddings, dtype="float32")

# Query embedding
def embed_query(query, model):
    return np.array(model.encode([query]), dtype="float32")

# Retrieval helper
def retrieve_top_k_with_query_embedding(query_embedding, chunks, index, k=1):
    distances, indices = index.search(query_embedding, k)
    
    results = []
    for rank, i in enumerate(indices[0], start=1):
        results.append({
            "rank": rank,
            "chunk": chunks[i],
            "distance": distances[0][rank - 1]
        })
    return results

# Evaluation
def evaluate_hit_at_k_local(evaluation_data, chunks, index, model, k=1):
    hits = 0
    
    for item in evaluation_data:
        query = item["query"]
        expected_keyword = item["expected_keyword"].lower()
        
        query_embedding = embed_query(query, model)
        results = retrieve_top_k_with_query_embedding(query_embedding, chunks, index, k=k)
        
        found = any(expected_keyword in result["chunk"].lower() for result in results)
        if found:
            hits += 1
    
    return hits / len(evaluation_data)

# Build indexes
minilm_chunk_embeddings = get_embeddings(section_chunks, minilm_model)
minilm_index = build_faiss_index_from_embeddings(minilm_chunk_embeddings)

bge_chunk_embeddings = get_embeddings(section_chunks, bge_model)
bge_index = build_faiss_index_from_embeddings(bge_chunk_embeddings)

# Compare
minilm_hit1 = evaluate_hit_at_k_local(evaluation_data, section_chunks, minilm_index, minilm_model, k=1)
minilm_hit2 = evaluate_hit_at_k_local(evaluation_data, section_chunks, minilm_index, minilm_model, k=2)

bge_hit1 = evaluate_hit_at_k_local(evaluation_data, section_chunks, bge_index, bge_model, k=1)
bge_hit2 = evaluate_hit_at_k_local(evaluation_data, section_chunks, bge_index, bge_model, k=2)

print("Embedding Comparison on Section-aware Chunks")
print("MiniLM Hit@1:", round(minilm_hit1, 2))
print("MiniLM Hit@2:", round(minilm_hit2, 2))
print("BGE-small Hit@1:", round(bge_hit1, 2))
print("BGE-small Hit@2:", round(bge_hit2, 2))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4286.91it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6124.33it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Comparison on Section-aware Chunks
MiniLM Hit@1: 1.0
MiniLM Hit@2: 1.0
BGE-small Hit@1: 1.0
BGE-small Hit@2: 1.0
